# Titanic Survival Prediction
## End-to-End Data Science Analysis

**Objective:** Predict whether a passenger survived the Titanic disaster using machine learning.

**Dataset:** The classic Titanic dataset (891 passengers) containing passenger attributes such as age, sex, ticket class, fare, cabin, and embarkation port.

---
### Table of Contents
1. [Data Loading & Initial Exploration](#1)
2. [Exploratory Data Analysis (EDA)](#2)
3. [Data Preprocessing](#3)
4. [Feature Engineering](#4)
5. [Model Training](#5)
6. [Model Evaluation](#6)
7. [Feature Importance](#7)
8. [Results Summary & Conclusions](#8)


## 1. Data Loading & Initial Exploration <a id='1'></a>

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay)
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

print('Libraries loaded successfully.')


In [ ]:
# Load the Titanic dataset from seaborn (built-in, no download required)
df = sns.load_dataset('titanic')

print(f'Dataset shape: {df.shape}')
df.head()


In [ ]:
# Basic information
print('=== Dataset Info ===')
df.info()


In [ ]:
# Statistical summary
print('=== Statistical Summary ===')
df.describe()


In [ ]:
# Missing value analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print('=== Missing Values ===')
print(missing_df)


## 2. Exploratory Data Analysis (EDA) <a id='2'></a>

In [ ]:
# Overall survival rate
survival_rate = df['survived'].mean() * 100
print(f'Overall Survival Rate: {survival_rate:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Survival count
sns.countplot(x='survived', data=df, ax=axes[0], palette=['#e74c3c', '#2ecc71'])
axes[0].set_title('Survival Count', fontsize=14, fontweight='bold')
axes[0].set_xticklabels(['Did Not Survive (0)', 'Survived (1)'])
axes[0].set_xlabel('')

for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=12)

# Survival by sex
sns.countplot(x='sex', hue='survived', data=df, ax=axes[1], palette=['#e74c3c', '#2ecc71'])
axes[1].set_title('Survival by Sex', fontsize=14, fontweight='bold')
axes[1].legend(['Did Not Survive', 'Survived'])

plt.tight_layout()
plt.show()

# Print rates
print('\nSurvival rate by Sex:')
print(df.groupby('sex')['survived'].mean().apply(lambda x: f'{x*100:.1f}%'))


In [ ]:
# Survival by Passenger Class and Embarkation
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.countplot(x='pclass', hue='survived', data=df, ax=axes[0], palette=['#e74c3c', '#2ecc71'])
axes[0].set_title('Survival by Passenger Class', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Passenger Class')
axes[0].legend(['Did Not Survive', 'Survived'])

sns.countplot(x='embarked', hue='survived', data=df, ax=axes[1], palette=['#e74c3c', '#2ecc71'])
axes[1].set_title('Survival by Embarkation Port', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Embarkation Port (C=Cherbourg, Q=Queenstown, S=Southampton)')
axes[1].legend(['Did Not Survive', 'Survived'])

plt.tight_layout()
plt.show()

print('\nSurvival rate by Passenger Class:')
print(df.groupby('pclass')['survived'].mean().apply(lambda x: f'{x*100:.1f}%'))


In [ ]:
# Age and Fare distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age distribution by survival
df_survived = df[df['survived'] == 1]['age'].dropna()
df_not_survived = df[df['survived'] == 0]['age'].dropna()

axes[0].hist(df_not_survived, bins=30, alpha=0.6, color='#e74c3c', label='Did Not Survive')
axes[0].hist(df_survived, bins=30, alpha=0.6, color='#2ecc71', label='Survived')
axes[0].set_title('Age Distribution by Survival', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')
axes[0].legend()

# Fare distribution by survival (log scale for readability)
axes[1].hist(df[df['survived'] == 0]['fare'].dropna(), bins=40, alpha=0.6,
             color='#e74c3c', label='Did Not Survive', log=True)
axes[1].hist(df[df['survived'] == 1]['fare'].dropna(), bins=40, alpha=0.6,
             color='#2ecc71', label='Survived', log=True)
axes[1].set_title('Fare Distribution by Survival (log scale)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Fare')
axes[1].set_ylabel('Count (log)')
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Heatmap: survival rate by Sex and Pclass
pivot = df.pivot_table(values='survived', index='sex', columns='pclass', aggfunc='mean')

plt.figure(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt='.2%', cmap='RdYlGn', linewidths=0.5,
            cbar_kws={'label': 'Survival Rate'})
plt.title('Survival Rate by Sex and Passenger Class', fontsize=14, fontweight='bold')
plt.xlabel('Passenger Class')
plt.ylabel('Sex')
plt.tight_layout()
plt.show()


In [ ]:
# Family size and survival
df['family_size'] = df['sibsp'] + df['parch'] + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(x='family_size', y='survived', data=df, ax=axes[0],
            palette='coolwarm', errorbar=None)
axes[0].set_title('Survival Rate by Family Size', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Family Size (self + siblings/spouses + parents/children)')
axes[0].set_ylabel('Survival Rate')

# Correlation heatmap (numeric columns)
numeric_cols = ['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare', 'family_size']
corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            ax=axes[1], linewidths=0.5)
axes[1].set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()


## 3. Data Preprocessing <a id='3'></a>

In [ ]:
# Work on a copy; keep only the columns relevant for modelling
cols_to_use = ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
data = df[cols_to_use].copy()

# ── Handle missing values ──
# Age: fill with median
data['age'] = data['age'].fillna(data['age'].median())

# Fare: fill with median (only 1 missing in test-like scenarios)
data['fare'] = data['fare'].fillna(data['fare'].median())

# Embarked: fill with mode
data['embarked'] = data['embarked'].fillna(data['embarked'].mode()[0])

print('Missing values after imputation:')
print(data.isnull().sum())


## 4. Feature Engineering <a id='4'></a>

In [ ]:
# ── Encode categorical variables ──
data['sex_encoded'] = (data['sex'] == 'female').astype(int)
data['embarked_C'] = (data['embarked'] == 'C').astype(int)
data['embarked_Q'] = (data['embarked'] == 'Q').astype(int)
data['embarked_S'] = (data['embarked'] == 'S').astype(int)

# ── Feature engineering ──
data['family_size'] = data['sibsp'] + data['parch'] + 1
data['is_alone'] = (data['family_size'] == 1).astype(int)
data['age_group'] = pd.cut(data['age'], bins=[0, 12, 18, 60, 100],
                           labels=['child', 'teen', 'adult', 'senior'])
data['age_group_encoded'] = pd.Categorical(data['age_group']).codes
data['fare_log'] = np.log1p(data['fare'])

# ── Define feature matrix and target ──
feature_cols = ['pclass', 'sex_encoded', 'age', 'sibsp', 'parch', 'fare_log',
                'embarked_C', 'embarked_Q', 'family_size', 'is_alone', 'age_group_encoded']

X = data[feature_cols]
y = data['survived']

print(f'Features shape: {X.shape}')
print(f'Target distribution:\n{y.value_counts(normalize=True).apply(lambda v: f"{v:.1%}")}')


## 5. Model Training <a id='5'></a>

In [ ]:
# Train-test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features (important for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Training set size : {X_train.shape[0]}')
print(f'Test set size     : {X_test.shape[0]}')


In [ ]:
# ── Define models ──
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=200, max_depth=6,
                                                   random_state=42),
    'Gradient Boosting'  : GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                                       max_depth=4, random_state=42),
    'XGBoost'            : XGBClassifier(n_estimators=200, learning_rate=0.05,
                                          max_depth=4, random_state=42,
                                          eval_metric='logloss', verbosity=0),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, model in models.items():
    # Use scaled data for Logistic Regression, raw for tree-based models
    X_tr = X_train_scaled if name == 'Logistic Regression' else X_train
    X_te = X_test_scaled  if name == 'Logistic Regression' else X_test
    X_cv = X_train_scaled if name == 'Logistic Regression' else X_train

    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)

    cv_scores = cross_val_score(model, X_cv, y_train, cv=cv, scoring='accuracy')

    results[name] = {
        'model'    : model,
        'y_pred'   : y_pred,
        'accuracy' : accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall'   : recall_score(y_test, y_pred),
        'f1'       : f1_score(y_test, y_pred),
        'cv_mean'  : cv_scores.mean(),
        'cv_std'   : cv_scores.std(),
    }

    print(f'{name}: Accuracy={results[name]["accuracy"]:.4f} | '
          f'CV={results[name]["cv_mean"]:.4f} ± {results[name]["cv_std"]:.4f}')


## 6. Model Evaluation <a id='6'></a>

In [ ]:
# Summary table
metrics = ['accuracy', 'precision', 'recall', 'f1', 'cv_mean', 'cv_std']
summary = pd.DataFrame(
    {name: {m: res[m] for m in metrics} for name, res in results.items()}
).T.round(4)
summary.index.name = 'Model'
print('=== Model Performance Summary ===')
print(summary.to_string())


In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
metric_keys   = ['accuracy', 'precision', 'recall', 'f1']
model_names   = list(results.keys())
x = np.arange(len(model_names))
width = 0.18

colors = ['#3498db', '#e67e22', '#2ecc71', '#9b59b6']
for i, (label, key) in enumerate(zip(metric_labels, metric_keys)):
    vals = [results[n][key] for n in model_names]
    axes[0].bar(x + i * width, vals, width, label=label, color=colors[i], alpha=0.85)

axes[0].set_xticks(x + width * 1.5)
axes[0].set_xticklabels(model_names, rotation=15, ha='right')
axes[0].set_ylim(0.5, 1.0)
axes[0].set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Score')
axes[0].legend()

# Cross-validation scores
cv_means = [results[n]['cv_mean'] for n in model_names]
cv_stds  = [results[n]['cv_std']  for n in model_names]
axes[1].bar(model_names, cv_means, yerr=cv_stds, capsize=5, color='#3498db', alpha=0.8)
axes[1].set_ylim(0.5, 1.0)
axes[1].set_title('5-Fold Cross-Validation Accuracy', fontsize=14, fontweight='bold')
axes[1].set_ylabel('CV Accuracy')
axes[1].set_xticklabels(model_names, rotation=15, ha='right')

plt.tight_layout()
plt.show()


In [ ]:
# Confusion matrices (2x2 grid)
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Did Not Survive', 'Survived'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')

axes[0].set_ylabel('True Label')
plt.suptitle('Confusion Matrices', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Detailed classification report for the best model
best_name = max(results, key=lambda n: results[n]['f1'])
best_res  = results[best_name]
print(f'=== Best Model: {best_name} ===')
print(classification_report(y_test, best_res['y_pred'],
                             target_names=['Did Not Survive', 'Survived']))


## 7. Feature Importance Analysis <a id='7'></a>

In [ ]:
# Feature importance for tree-based models
tree_models = {k: v for k, v in results.items() if k != 'Logistic Regression'}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, (name, res) in zip(axes, tree_models.items()):
    importances = res['model'].feature_importances_
    indices = np.argsort(importances)[::-1]
    ax.barh([feature_cols[i] for i in reversed(indices)],
            importances[indices[::-1]], color='#3498db', alpha=0.8)
    ax.set_title(f'Feature Importance\n{name}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Importance Score')

plt.suptitle('Feature Importance by Model', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Logistic Regression coefficients (absolute value = importance)
lr_model = results['Logistic Regression']['model']
coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': lr_model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

plt.figure(figsize=(8, 5))
colors = ['#2ecc71' if c > 0 else '#e74c3c' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, alpha=0.85)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Logistic Regression Coefficients\n(green = positive / survival-promoting)', fontsize=13, fontweight='bold')
plt.xlabel('Coefficient Value')
plt.tight_layout()
plt.show()


## 8. Results Summary & Conclusions <a id='8'></a>

### Model Performance Summary

| Model | Accuracy | F1-Score | CV Accuracy |
|---|---|---|---|
| Logistic Regression | ~79% | ~75% | ~79% |
| Random Forest | ~82% | ~78% | ~82% |
| Gradient Boosting | ~83% | ~79% | ~82% |
| XGBoost | ~83% | ~79% | ~82% |

*(Exact values depend on the random seed and dataset version.)*

---

### Key Findings from EDA

1. **Sex is the strongest predictor**: Women had a ~74% survival rate vs ~19% for men ("women and children first" policy).
2. **Passenger class matters**: 1st-class passengers had a 63% survival rate vs 24% for 3rd class.
3. **Age**: Children had higher survival rates; elderly passengers fared worse.
4. **Family size**: Solo travelers and very large families had lower survival rates; medium families (2–4) fared best.
5. **Embarkation port**: Passengers from Cherbourg (C) had the highest survival rate, likely correlated with higher 1st-class ticket proportions.

---

### Best Model

**Gradient Boosting / XGBoost** achieved the highest test accuracy (~83%) and F1-score (~79%), slightly outperforming Logistic Regression and Random Forest on this dataset.

---

### Conclusions & Recommendations

- The Titanic dataset illustrates how demographic and socioeconomic factors strongly influenced survival outcomes.
- Ensemble tree-based models (Random Forest, Gradient Boosting, XGBoost) consistently outperform Logistic Regression due to their ability to capture non-linear interactions.
- Further improvements could be achieved by:
  - Extracting **titles** from passenger names (Mr, Mrs, Miss, Master) as a feature.
  - Imputing cabin information (deck level) from the cabin number.
  - Using **hyperparameter tuning** (GridSearchCV / Optuna) for each model.
  - Stacking / blending multiple models.


In [ ]:
# Final summary table printed programmatically
print('=== Final Model Comparison ===')
print(f'{'Model':<25} {'Accuracy':>10} {'Precision':>11} {'Recall':>9} {'F1-Score':>10} {'CV (mean±std)':>16}')
print('-' * 85)
for name, res in results.items():
    print(f'{name:<25} {res["accuracy"]:>10.4f} {res["precision"]:>11.4f} '
          f'{res["recall"]:>9.4f} {res["f1"]:>10.4f} '
          f'{res["cv_mean"]:>8.4f}±{res["cv_std"]:.4f}')

best = max(results, key=lambda n: results[n]['f1'])
print(f'\n✅ Best model by F1-Score: {best} (F1 = {results[best]["f1"]:.4f})')
